In [8]:
!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install transformers
!pip -q install torch
!pip -q install pypdf

In [9]:
import numpy as np
import faiss
import torch

from google.colab import files

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

In [10]:
uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Uploaded:", pdf_file)

Saving Unit 4-Transport Layer.pdf to Unit 4-Transport Layer.pdf
Uploaded: Unit 4-Transport Layer.pdf


In [11]:
reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    if page.extract_text():
        text += page.extract_text()

print(text[:1000])

Unit 4
Transport Layer
Ms. Shweta Paliwal
Assistant Professor -SoCIntroduction
•The transport layer, orlayer 4oftheOSI model, controls network
traffic between hosts andendsystems toguarantee fulldata flows .
•The Transport Layer isresponsible forend-to-endcommunication of
data packets .Services Offered By Transport Layer
(a)End toEnd Communication :Thetransport layer isresponsible forcreating the
end-to-endConnection between hosts forwhich itmainly uses TCP andUDP .
•TCP isasecure, connection -orientated protocol thatuses ahandshake protocol to
establish arobust connection between two end hosts and ensures thereliable
delivery ofmessages andisused invarious applications .
•UDP ,ontheother hand, isastateless andunreliable protocol that ensures best-
effort delivery .Itissuitable forapplications thathave little concern with flow or
error control andrequires sending thebulk ofdata likevideo conferencing .Itis
often used inmulticasting protocols•Flow Control : The transport layer provides 

In [12]:
chunk_size = 500

chunks = []

for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i+chunk_size])

print("Total Chunks:", len(chunks))

Total Chunks: 76


In [13]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [14]:
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

print(embeddings.shape)

(76, 384)


In [15]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Vectors Stored:", index.ntotal)

Vectors Stored: 76


In [16]:
tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [17]:
def retrieve(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    retrieved = []

    for idx in indices[0]:
        retrieved.append(chunks[idx])

    return "\n\n".join(retrieved)

In [18]:
def ask(query):

    context = retrieve(query)

    prompt = f"""
Answer ONLY using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("="*60)
    print("QUESTION")
    print("="*60)
    print(query)

    print()

    print("="*60)
    print("ANSWER")
    print("="*60)
    print(answer)

In [ ]:
while True:

    q = input("\nAsk a question (exit to quit): ")

    if q.lower() == "exit":
        break

    ask(q)


Ask a question (exit to quit): what is the document about?
QUESTION
what is the document about?

ANSWER
telecommunications

Ask a question (exit to quit): who made this document?
QUESTION
who made this document?

ANSWER
White

Ask a question (exit to quit): who created this document?
QUESTION
who created this document?

ANSWER
the server

Ask a question (exit to quit): what is the 4th layer in telecommunication?
QUESTION
what is the 4th layer in telecommunication?

ANSWER
transport layer

Ask a question (exit to quit): what is transport layer?
QUESTION
what is transport layer?

ANSWER
controls network traffic between hosts and endsystems toguarante fulldata flows

Ask a question (exit to quit): summarize the document for me.
QUESTION
summarize the document for me.

ANSWER
Step 2 (SYN + ACK)

Ask a question (exit to quit): what are the names of all layers?
QUESTION
what are the names of all layers?

ANSWER
unit 4 transport layer

Ask a question (exit to quit): how long is the pdf?
QUES